# Viewer 3.1 — presentation exports and NAV controls

This notebook derives `viewer/mouse_lung_v3.1.html` from the committed 3.0 viewer. It changes presentation controls and documentation, **not** the deposited M07 or E02 geometry. M07 coordinates remain local to that model. Ruberte et al. (2025) supports standardized quadruped anatomical **terminology**, not M07 dimensions or Xenium-to-M07 registration.

GIF recording samples the WebGL view at about 12 frames/s for at most 6 seconds; quantization and encoding run in the visitor's browser. PNG preserves full alpha. GIF has one-bit transparency. Both outputs omit the left control panel and retain a small orientation compass and, when shown, concise plane settings.

The GIF encoder is inlined into the generated HTML because local `file://` pages can block imports of adjacent module files. Its vendored source and license remain under `assets/`.

Version 3.1 now presents three explicitly bounded GIF capture profiles before recording. Width and frame rate increase visual detail but also raise client-side memory and encoding cost; none changes M07 geometry.

The M07 lobe layer starts at 95% opacity on each fresh load; the range control and material opacity are synchronized by dispatching the normal input event after the meshes exist.

The GIF has no fixed recording duration. Its encoder receives frames as they are captured, so raw frames do not accumulate in memory. The GIF byte stream still grows with recording length. When clipping is active at the start, the output uses a 16:9 scientific slide layout with the 3D view, actual 2D mesh section, NAV orientation and exact M07-local plane address.
The main plane card now keeps only its name, tilt and offset. The live NAV coordinates move to Info, alongside provenance and an acknowledgment of Abel Lopez. The section caption describes the actual M07 mesh intersection without naming a downstream assay.

GIF exports use the viewer background color and include a discreet creator/source credit; the MIT license remains in the repository. A Homepage control links to the verified public site. PNG retains transparency.

Journal-oriented exports use a 3200 × 1800 opaque Figure PNG with a 2200-pixel transparent alternative, both bearing a small citation-style creator credit and version. Supplementary video is browser-encoded as H.264 MP4 when available, or WebM with an explicit warning. The normal-offset drag pad is keyboard accessible. Journal acceptance and size limits require separate verification.


In [1]:
from pathlib import Path
import hashlib, re

ROOT = Path.cwd()
assert (ROOT / 'viewer/archive/mouse_lung_v3.0.html').exists(), 'Run from the repository root'
src = (ROOT / 'viewer/archive/mouse_lung_v3.0.html').read_text()
def replace_once(old, new):
    global src
    count = src.count(old)
    assert count == 1, (old[:80], count)
    src = src.replace(old, new, 1)

print('Source bytes:', len(src.encode()), 'SHA256:', hashlib.sha256(src.encode()).hexdigest())

Source bytes: 44874700 SHA256: fc602ada64d9de2ccb23c135fc41e30e93c139a8ac6de944fa59c74335ae76ea


## 1. Presentation toolbar and compact plane controls
The icon buttons retain accessible names and keyboard focus. Exact numeric inputs supplement native sliders; the 2D drag pad changes azimuth and elevation together.

In [2]:
replace_once('<title>Mouse Lung Explorer 3.0 | anatomical reference</title>', '<title>Mouse Lung Explorer 3.1 | M07 model · NAV anatomy</title>')
replace_once('3.0 · C57BL/6 m07 anatomical reference', '3.1 · M07 model · NAV / Ruberte 2025')
replace_once('Mouse lung explorer · 3.0', 'Mouse lung explorer · 3.1')
replace_once('<output id="opacityOut">100%</output></label><input id="opacity" type="range" min="15" max="100" value="100"', '<output id="opacityOut">95%</output></label><input id="opacity" type="range" min="15" max="100" value="95"')
replace_once("  document.getElementById('airwayVisible').addEventListener('change'", "  opacity.dispatchEvent(new Event('input'));\n  document.getElementById('airwayVisible').addEventListener('change'")
replace_once("for(const id of ['depth','angle','angle2'])cutBody.appendChild(document.getElementById(id).closest('.control'));", "cutBody.appendChild(ui.querySelector('.planeControls'));")
replace_once("address.querySelector('.small').textContent='M07 frame: right, ventral, cranial. az = horizontal angle; el = elevation; d = signed distance from the model origin.';", "address.querySelector('.small').textContent='M07 local model frame (mm): right, ventral, cranial per NAV terminology discussed by Ruberte et al. (2025). az and el are plane angles; d is signed distance from the M07 origin. These are not Xenium coordinates.';")
replace_once('<button id="saveLungPng" type="button" class="captureButton" aria-label="Save lung view as PNG">Save PNG</button>\n  <button id="resumeRotateBtn" type="button">Resume rotation</button>', """<div id="presentationToolbar" role="toolbar" aria-label="Presentation export and motion">
    <button id="saveLungPng" type="button" class="iconButton" aria-label="Download transparent PNG" title="Download transparent PNG"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M4 7h3l1.5-2h7L17 7h3v12H4z"/><circle cx="12" cy="13" r="3.2"/></svg></button>
    <button id="recordLungGif" type="button" class="iconButton" aria-label="Choose GIF recording quality" title="Choose GIF recording quality"><svg viewBox="0 0 24 24" aria-hidden="true"><rect x="3" y="5" width="18" height="14" rx="2"/><circle cx="12" cy="12" r="3.1" fill="currentColor" stroke="none"/></svg></button><span id="gifTimer" hidden aria-hidden="true">00:00</span>
    <button id="resumeRotateBtn" type="button" class="iconButton" aria-label="Resume automatic rotation" title="Resume or pause automatic rotation"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M20 8a8 8 0 1 0 1 6M20 3v5h-5"/></svg></button>
    <button id="rotationSpeed" type="button" class="speedButton" aria-label="Rotation speed 1 times" title="Change automatic rotation speed">1×</button>
  </div>
  <div id="gifSetup" class="panel" role="dialog" aria-label="GIF recording settings" hidden>
    <strong>Record GIF</strong><button id="gifCancel" type="button" aria-label="Close GIF settings" title="Close">×</button>
    <label for="gifQuality">Quality</label>
    <select id="gifQuality" aria-label="GIF capture quality">
      <option value="compact">Compact · 640 px · 12 fps</option>
      <option value="clear">Clear · 960 px · 15 fps</option>
      <option value="slide" selected>Slide · 1280 px · 20 fps</option>
    </select>
    <p id="gifEstimate" class="small" aria-live="polite"></p>
    <button id="gifStart" type="button" class="primary">Start recording</button>
  </div>""")
replace_once('<div class="control"><label for="depth"><span>Plane offset along its normal</span><output id="depthOut">0.00</output></label><input id="depth" type="range" min="-10" max="10" value="0" step="0.05" /></div>\n    <div class="control"><label for="angle"><span>Plane azimuth · LR ↔ DV</span><output id="angleOut">15°</output></label><input id="angle" type="range" min="-90" max="90" value="15" step="1" /></div>\n    <div class="control"><label for="angle2"><span>Plane elevation · Cr ↔ Cd</span><output id="angle2Out">0°</output></label><input id="angle2" type="range" min="-90" max="90" value="0" step="1" /></div>', """<div class="planeControls" aria-label="NAV-oriented model plane controls">
      <div class="planeControlHead"><strong>Section plane</strong><span>M07 frame · mm</span></div>
      <div class="planeControlRow"><label for="depth">Offset <small>normal distance</small></label><input id="depth" type="range" min="-10" max="10" value="0" step="0.05"/><input id="depthNumber" type="number" min="-10" max="10" step="0.05" value="0" aria-label="Plane offset in millimeters"/><span class="unit">mm</span><output id="depthOut" class="srOnly">0.00</output></div>
      <div class="planeControlRow"><label for="angle">Azimuth <small>right ↔ ventral</small></label><input id="angle" type="range" min="-90" max="90" value="15" step="1"/><input id="angleNumber" type="number" min="-90" max="90" step="1" value="15" aria-label="Azimuth in degrees"/><span class="unit">°</span><output id="angleOut" class="srOnly">15°</output></div>
      <div class="planeControlRow"><label for="angle2">Elevation <small>cranial ↔ caudal</small></label><input id="angle2" type="range" min="-90" max="90" value="0" step="1"/><input id="angle2Number" type="number" min="-90" max="90" step="1" value="0" aria-label="Elevation in degrees"/><span class="unit">°</span><output id="angle2Out" class="srOnly">0°</output></div>
      <div class="tiltPadWrap"><span>Drag to tilt</span><div id="tiltPad" role="application" aria-label="Drag to set azimuth and elevation; keyboard use numeric fields"><i id="tiltHandle"></i></div><span class="padHint">azimuth → · elevation ↑</span></div>
    </div>""")
replace_once("alveolarButton.textContent=\'Alveolar ROI · 3D + section\'", "alveolarButton.textContent='Alveolar chicken-wire view';alveolarButton.title='Explore an independent measured alveolar-region ROI (not registered to M07)'")
replace_once('Reference lung frame: +X right, +Y ventral, +Z cranial; mm. Plane n·x = d. Origin belongs to this exact mesh, not the body or Xenium slide.', 'M07 local frame (mm): +X right, +Y ventral, +Z cranial; plane n·x = d. Direction names follow NAV terminology discussed by Ruberte et al. (2025). The origin and dimensions belong to this M07 mesh, not to Ruberte et al. or Xenium.')
replace_once("const renderer=new THREE.WebGLRenderer({antialias:true,alpha:false,stencil:true,preserveDrawingBuffer:true,powerPreference:'high-performance'});", "const renderer=new THREE.WebGLRenderer({antialias:true,alpha:true,stencil:true,preserveDrawingBuffer:true,powerPreference:'high-performance'});")
replace_once("controls.autoRotateSpeed=1.00; let currentView", "controls.autoRotateSpeed=1.00; let currentView")
replace_once("function setResumeButton(){resumeRotateBtn.style.display=autoRotateStoppedByUser?'block':'none';}", "function setResumeButton(){resumeRotateBtn.setAttribute('aria-pressed',String(!autoRotateStoppedByUser));resumeRotateBtn.title=autoRotateStoppedByUser?'Resume automatic rotation':'Pause automatic rotation';}")
replace_once("resumeRotateBtn.addEventListener('click',()=>resumeAutoRotate());", "resumeRotateBtn.addEventListener('click',()=>autoRotateStoppedByUser?resumeAutoRotate():stopAutoRotateByUser());\n  const speedButton=document.getElementById('rotationSpeed');const speedValues=[1,1.5,2];let speedIndex=0;speedButton.addEventListener('click',()=>{speedIndex=(speedIndex+1)%speedValues.length;controls.autoRotateSpeed=speedValues[speedIndex];speedButton.textContent=speedValues[speedIndex]+'×';speedButton.setAttribute('aria-label','Rotation speed '+speedValues[speedIndex]+' times');});setResumeButton();")
replace_once('      <details class="miniDetails">\n        <summary>Model coordinates (mm) & NAV base</summary>\n        <div class="miniBody">\n          <div id="planeNavBase" class="planeLine"></div>\n          <div id="planeNormal" class="planeMono"></div>\n          <div id="planePoint" class="planeMono"></div>\n        </div>\n      </details>', '')
replace_once('Illustrative reference plane · not D14 / D30', 'M07 model plane')
replace_once('    <div class="planeCard" style="margin-top:10px">\n      <div class="planeKicker">Sources</div>', '    <div class="planeCard" style="margin-top:10px">\n      <div class="planeKicker">Current M07 plane · model coordinates (mm)</div>\n      <div class="small">This address uses the local M07 model frame: +X right, +Y ventral, +Z cranial. The plane equation is n · x = d. These coordinates describe the reference mesh and are not registered to another specimen.</div>\n      <div id="planeNavBase" class="planeLine"></div>\n      <div id="planeNormal" class="planeMono"></div>\n      <div id="planePoint" class="planeMono"></div>\n    </div>\n    <div class="planeCard" style="margin-top:10px">\n      <div class="planeKicker">Sources</div>')
replace_once('        <li><b>Xenium section evidence.</b> de Lima et al., Immunity (2026), doi:10.1016/j.immuni.2026.01.023. GEO GSM8837316 (D14) and GSM9364368 (D30): 5 µm FFPE sections. Lobe, plane, depth and embedding orientation are not recorded in the inspected methods or sample descriptions.</li>\n', '')
replace_once('      </ol>\n    </div>\n  </aside>\n  <aside id="slicePanel"', '      </ol>\n    </div>\n    <div class="planeCard" style="margin-top:10px">\n      <div class="planeKicker">Acknowledgments</div>\n      <div class="small">Thank you to <a href="https://github.com/alopez789" target="_blank" rel="noopener noreferrer">Abel Lopez</a> for helping build the 3D lung and airway model.</div>\n    </div>\n  </aside>\n  <aside id="slicePanel"')
replace_once('This is an anatomical mesh section, not simulated IF/Xenium: cells and expression are unavailable.', 'This 2D view intersects the M07 lobe and airway surface meshes at the selected plane; it does not contain cellular detail.')
replace_once("document.querySelector('.planeKicker').textContent='Reference plane — not a measured D14 / D30 registration';", "document.querySelector('.planeKicker').textContent='M07 model plane';")
replace_once('<button id="toggleInfo" class="ghost">Info</button>', '<button id="toggleInfo" class="ghost">Info</button>\n      <a id="homeLink" class="homeButton" href="https://arashabadi.github.io/" target="_blank" rel="noopener noreferrer" aria-label="Open Arash Bagherabadi homepage">Homepage ↗</a>')
replace_once('<button id="resumeRotateBtn" type="button" class="iconButton"', '<button id="recordLungVideo" type="button" class="iconButton" aria-label="Set up supplementary video" title="Set up supplementary video"><svg viewBox="0 0 24 24" aria-hidden="true"><rect x="3" y="6" width="12" height="12" rx="2"/><path d="m15 10 6-3v10l-6-3z"/></svg></button>\n    <button id="resumeRotateBtn" type="button" class="iconButton"')
replace_once('  <div id="gifSetup" class="panel" role="dialog"', '  <div id="pngSetup" class="panel" role="dialog" aria-label="PNG export settings" hidden>\n    <strong>Export PNG</strong><button id="pngCancel" type="button" aria-label="Close PNG settings">×</button>\n    <label for="pngQuality">Output</label><select id="pngQuality" aria-label="PNG output style"><option value="figure" selected>Figure · 3200 px · dark 16:9</option><option value="transparent">Transparent · 2200 px</option></select>\n    <p class="small">Both outputs include the creator credit. Figure mode includes the 2D section when Clip is active.</p>\n    <button id="pngExport" type="button" class="primary">Download PNG</button>\n  </div>\n  <div id="gifSetup" class="panel" role="dialog"')
replace_once('  <div id="sideHint">', '  <div id="videoSetup" class="panel" role="dialog" aria-label="Supplementary video settings" hidden>\n    <strong>Record supplementary video</strong><button id="videoCancel" type="button" aria-label="Close video settings">×</button>\n    <label for="videoQuality">Resolution</label><select id="videoQuality" aria-label="Video resolution"><option value="1920" selected>Full HD · 1920 × 1080 · 30 fps</option><option value="1280">HD · 1280 × 720 · 30 fps</option></select>\n    <p id="videoFormatNote" class="small" aria-live="polite"></p>\n    <button id="videoStart" type="button" class="primary">Start video</button>\n  </div>\n  <div id="sideHint">')
replace_once('<span class="padHint">azimuth → · elevation ↑</span></div>\n    </div>', '<span class="padHint">azimuth → · elevation ↑</span></div>\n      <div class="offsetPadWrap"><span>Slide plane</span><div id="offsetPad" role="slider" tabindex="0" aria-label="Drag to move section plane along its normal" aria-valuemin="-10" aria-valuemax="10" aria-valuenow="0"><i id="offsetHandle"></i></div><output id="offsetPadValue">0.00 mm</output></div>\n    </div>')
replace_once('Mouse lung explorer · 3.1', 'Mouse lung explorer · v3.1')
replace_once("ui.querySelector('.sub').textContent='Reference anatomy · explore a plane or open an independent alveolar ROI.';", "ui.querySelector('.sub').textContent='Reference anatomy · explore a plane or open an independent alveolar ROI.';ui.querySelector('.eyebrow').remove();")
replace_once("ui.insertBefore(alveolarButton,cuts);", "ui.insertBefore(alveolarButton,cuts);const sideHint=document.getElementById('sideHint');ui.insertBefore(sideHint,cuts);")
print('Toolbar and plane controls patched')

Toolbar and plane controls patched


In [3]:
style = r"""
/* v3.1: deliberately quiet presentation controls */
#presentationToolbar{position:absolute;right:16px;top:16px;z-index:45;display:flex;gap:6px;padding:5px;border:1px solid rgba(124,216,255,.24);border-radius:16px;background:rgba(7,16,24,.85);backdrop-filter:blur(14px)}
#presentationToolbar button{position:static;display:grid;place-items:center;min-width:36px;width:36px;height:36px;min-height:36px;padding:6px;border-radius:11px;background:transparent;border:1px solid transparent;color:#dbf8ff;box-shadow:none}
#presentationToolbar button:hover,#presentationToolbar button[aria-pressed="true"]{border-color:rgba(124,216,255,.45);background:rgba(124,216,255,.13)}
#presentationToolbar button:focus-visible,.planeControlRow input:focus-visible,#tiltPad:focus-visible{outline:2px solid #83d7ff;outline-offset:2px}
#presentationToolbar svg{width:19px;height:19px;fill:none;stroke:currentColor;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round}
#presentationToolbar .speedButton{font-size:11px;font-weight:750;letter-spacing:-.02em}
#presentationToolbar #gifTimer{align-self:center;min-width:36px;color:#ff9999;font:700 10px ui-monospace,SFMono-Regular,monospace}#presentationToolbar #gifTimer[hidden]{display:none}
#presentationToolbar #recordLungGif[aria-pressed="true"]{color:#ff7474;border-color:#ff7474;background:rgba(255,85,85,.17)}
#gifSetup{position:absolute;right:16px;top:72px;z-index:47;width:min(294px,calc(100vw - 32px));padding:14px;background:rgba(8,17,25,.96);border-color:rgba(124,216,255,.35)}
#gifSetup[hidden]{display:none}#gifSetup strong{font-size:12px}#gifSetup #gifCancel{float:right;min-width:26px;width:26px;height:26px;min-height:26px;padding:0;font-size:19px;line-height:1}
#gifSetup label{display:block;margin-top:12px;margin-bottom:5px;color:var(--muted);font-size:10px}#gifQuality{width:100%;padding:8px;color:var(--text);background:#12222e;border:1px solid rgba(124,216,255,.35);border-radius:8px;font:11px system-ui,sans-serif}
#gifEstimate{min-height:43px;margin:9px 0;color:var(--muted)}#gifStart{width:100%}#sideHint{top:77px}
.srOnly{position:absolute!important;width:1px;height:1px;padding:0;margin:-1px;overflow:hidden;clip:rect(0,0,0,0);white-space:nowrap;border:0}
.planeControls{margin:12px 0 10px;padding:11px;border:1px solid rgba(124,216,255,.2);background:rgba(7,18,28,.62);border-radius:14px}
.planeControlHead{display:flex;align-items:baseline;justify-content:space-between;margin-bottom:7px;font-size:11px}.planeControlHead span{font-size:9px;color:var(--muted)}
.planeControlRow{display:grid;grid-template-columns:85px minmax(45px,1fr) 54px 16px;gap:7px;align-items:center;min-height:43px;border-top:1px solid rgba(255,255,255,.06)}
.planeControlRow label{font-size:10.5px;color:#e7f6fc}.planeControlRow small{display:block;font-size:8.5px;line-height:1.2;color:var(--muted)}
.planeControlRow input[type="range"]{width:100%;accent-color:#80d9fc;margin:0;cursor:ew-resize}
.planeControlRow input[type="number"]{width:54px;height:27px;padding:2px 3px;color:#eafaff;background:rgba(124,216,255,.07);border:1px solid rgba(124,216,255,.26);border-radius:7px;text-align:right;font:10px ui-monospace,SFMono-Regular,monospace;appearance:textfield}
.planeControlRow input[type="number"]::-webkit-inner-spin-button{-webkit-appearance:none;margin:0}.planeControlRow .unit{color:#96a9b8;font-size:10px}
.tiltPadWrap{display:flex;align-items:center;gap:8px;margin-top:8px;color:var(--muted);font-size:9px}.tiltPadWrap>span:first-child{min-width:69px}
#tiltPad{position:relative;width:72px;height:72px;flex:none;border-radius:10px;border:1px solid rgba(124,216,255,.28);background:linear-gradient(90deg,transparent calc(50% - .5px),rgba(124,216,255,.16) 50%,transparent calc(50% + .5px)),linear-gradient(transparent calc(50% - .5px),rgba(124,216,255,.16) 50%,transparent calc(50% + .5px)),rgba(124,216,255,.035);touch-action:none;cursor:crosshair}
#tiltHandle{position:absolute;width:11px;height:11px;border-radius:50%;background:#83d7ff;box-shadow:0 0 0 4px rgba(124,216,255,.15);transform:translate(-50%,-50%);left:50%;top:50%;pointer-events:none}.padHint{line-height:1.35}
@media(max-width:700px){#presentationToolbar{right:10px;top:10px;gap:2px}#presentationToolbar button{width:32px;height:32px;min-width:32px}#sideHint{top:63px}}
"""
style += '''
#ui .homeButton{display:flex;align-items:center;justify-content:center;min-height:32px;margin-top:8px;border:1px solid rgba(124,216,255,.24);border-radius:9px;background:rgba(124,216,255,.06);color:#b9e8f4;text-decoration:none;font-size:10.5px;font-weight:650}
#ui .homeButton{grid-column:1/-1}#ui .homeButton:hover{border-color:#7cd8ff;background:rgba(124,216,255,.13)}#ui .homeButton:focus-visible{outline:2px solid #83d7ff;outline-offset:2px}
'''
style += '''
#pngSetup,#videoSetup{position:absolute;right:16px;top:72px;z-index:47;width:min(315px,calc(100vw - 32px));padding:14px;background:rgba(8,17,25,.96);border-color:rgba(124,216,255,.35)}
#pngSetup[hidden],#videoSetup[hidden]{display:none}#pngSetup strong,#videoSetup strong{font-size:12px}
#pngSetup #pngCancel,#videoSetup #videoCancel{float:right;min-width:26px;width:26px;height:26px;min-height:26px;padding:0;font-size:19px;line-height:1}
#pngSetup label,#videoSetup label{display:block;margin-top:12px;margin-bottom:5px;color:var(--muted);font-size:10px}
#pngQuality,#videoQuality{width:100%;padding:8px;color:var(--text);background:#12222e;border:1px solid rgba(124,216,255,.35);border-radius:8px;font:11px system-ui,sans-serif}
#pngExport,#videoStart{width:100%}#presentationToolbar #recordLungVideo[aria-pressed="true"]{color:#ff7474;border-color:#ff7474;background:rgba(255,85,85,.17)}
'''
style += '''
.offsetPadWrap{display:grid;grid-template-columns:69px minmax(90px,1fr) 55px;align-items:center;gap:8px;margin-top:11px;color:var(--muted);font-size:9px}
#offsetPad{position:relative;height:48px;min-width:100px;border:1px solid rgba(124,216,255,.28);border-radius:12px;background:linear-gradient(90deg,transparent calc(50% - .5px),rgba(124,216,255,.28) 50%,transparent calc(50% + .5px)),linear-gradient(rgba(124,216,255,.03),rgba(124,216,255,.08));touch-action:none;cursor:ew-resize}
#offsetPad::before{content:'−10';position:absolute;left:9px;top:6px;color:#718c9a}#offsetPad::after{content:'+10';position:absolute;right:9px;top:6px;color:#718c9a}
#offsetHandle{position:absolute;top:29px;left:50%;width:17px;height:17px;border-radius:50%;background:#83d7ff;box-shadow:0 0 0 5px rgba(124,216,255,.16);transform:translate(-50%,-50%);pointer-events:none}
#offsetPadValue{text-align:right;color:#d7edf5;font:10px ui-monospace,SFMono-Regular,monospace}
#offsetPad:focus-visible{outline:2px solid #83d7ff;outline-offset:2px}
'''
style += '''
#ui #sideHint{position:static;width:auto;max-width:none;top:auto;right:auto;margin:9px 0 2px;padding:8px 10px;font-size:10.4px;line-height:1.35}
#orientation{width:245px;padding:13px 14px;font-size:11.5px}#orientationSvg{width:175px;height:175px}
#orientation>div:first-child{font-size:12.5px!important}
'''
replace_once('</style>', style+'</style>')

numeric_js = r"""
  // Exact values remain synchronized after slider, pad, link-restore or preset changes.
  const planeNumbers=[['depth','depthNumber',2],['angle','angleNumber',1],['angle2','angle2Number',1]];
  for(const [sliderId,numberId,precision] of planeNumbers){
    const slider=document.getElementById(sliderId),number=document.getElementById(numberId);
    slider.addEventListener('input',()=>{number.value=Number(slider.value).toFixed(precision)});
    number.addEventListener('input',()=>{const value=Math.max(Number(slider.min),Math.min(Number(slider.max),Number(number.value)));if(Number.isFinite(value)){slider.value=value;slider.dispatchEvent(new Event('input',{bubbles:true}))}});
  }
  const tiltPad=document.getElementById('tiltPad'),tiltHandle=document.getElementById('tiltHandle'),offsetPad=document.getElementById('offsetPad'),offsetHandle=document.getElementById('offsetHandle'),offsetPadValue=document.getElementById('offsetPadValue');
  function syncPlaneControls(){for(const [sliderId,numberId,precision] of planeNumbers){const slider=document.getElementById(sliderId),number=document.getElementById(numberId);if(document.activeElement!==number)number.value=Number(slider.value).toFixed(precision)}tiltHandle.style.left=((Number(angle.value)+90)/180*100)+'%';tiltHandle.style.top=((90-Number(angle2.value))/180*100)+'%';offsetHandle.style.left=((Number(depth.value)+10)/20*100)+'%';offsetPadValue.value=Number(depth.value).toFixed(2)+' mm';offsetPad.setAttribute('aria-valuenow',Number(depth.value).toFixed(2))}
  function moveTilt(event){const r=tiltPad.getBoundingClientRect(),x=Math.max(0,Math.min(1,(event.clientX-r.left)/r.width)),y=Math.max(0,Math.min(1,(event.clientY-r.top)/r.height));angle.value=Math.round(x*180-90);angle2.value=Math.round(90-y*180);angle.dispatchEvent(new Event('input',{bubbles:true}));angle2.dispatchEvent(new Event('input',{bubbles:true}));syncPlaneControls()}
  tiltPad.addEventListener('pointerdown',event=>{tiltPad.setPointerCapture(event.pointerId);moveTilt(event)});
  tiltPad.addEventListener('pointermove',event=>{if(tiltPad.hasPointerCapture(event.pointerId))moveTilt(event)});
  let offsetStartX=0,offsetStartValue=0;
  function moveOffset(event){const span=Number(depth.max)-Number(depth.min),delta=(event.clientX-offsetStartX)/offsetPad.clientWidth*span*(event.shiftKey?.15:1),value=Math.max(Number(depth.min),Math.min(Number(depth.max),offsetStartValue+delta));depth.value=value.toFixed(2);depth.dispatchEvent(new Event('input',{bubbles:true}));syncPlaneControls()}
  offsetPad.addEventListener('pointerdown',event=>{offsetStartX=event.clientX;offsetStartValue=Number(depth.value);offsetPad.setPointerCapture(event.pointerId)});
  offsetPad.addEventListener('pointermove',event=>{if(offsetPad.hasPointerCapture(event.pointerId))moveOffset(event)});
  offsetPad.addEventListener('keydown',event=>{if(!['ArrowLeft','ArrowRight','Home','End'].includes(event.key))return;event.preventDefault();const value=event.key==='Home'?-10:event.key==='End'?10:Math.max(-10,Math.min(10,Number(depth.value)+(event.key==='ArrowRight'?1:-1)*(event.shiftKey?.01:.1)));depth.value=value.toFixed(2);depth.dispatchEvent(new Event('input',{bubbles:true}));syncPlaneControls()});
  syncPlaneControls();
"""
replace_once('  // v26 visibility aid and unregistered source-section panel.', numeric_js+'\n  // v26 visibility aid and unregistered source-section panel.')
print('CSS and fine plane interactions patched')

CSS and fine plane interactions patched


## 2. Transparent image and browser-side GIF
The WebGL scene is rendered with an alpha channel only for capture, then restored. The orientation overlay is drawn from the live NAV compass geometry; browser panels are never copied into the export. GIF is palette-quantized with a single transparent index, so soft alpha edges cannot survive.

In [4]:
capture_js = r"""
  // Export the geometry canvas, not a screenshot of page controls.
  const capturePngButton=document.getElementById('saveLungPng');
  const gifButton=document.getElementById('recordLungGif');
  function drawCompactCompass(ctx,w,h,onLeft=false){
    const svg=document.getElementById('orientationSvg'),size=Math.max(105,Math.min(300,w*.18)),x=onLeft?18:w-size-18,y=h-size-22,scale=size/128;
    ctx.save();ctx.translate(x,y);ctx.scale(scale,scale);
    for(const line of svg.querySelectorAll('line')){ctx.beginPath();ctx.moveTo(Number(line.getAttribute('x1')),Number(line.getAttribute('y1')));ctx.lineTo(Number(line.getAttribute('x2')),Number(line.getAttribute('y2')));ctx.strokeStyle=line.getAttribute('stroke')||'#d7f3ff';ctx.globalAlpha=Number(line.getAttribute('opacity')||1);ctx.lineWidth=2.5;ctx.lineCap='round';ctx.stroke()}
    ctx.globalAlpha=1;ctx.fillStyle='#d7f3ff';ctx.beginPath();ctx.arc(64,64,3,0,Math.PI*2);ctx.fill();ctx.font='bold 10px system-ui,sans-serif';ctx.textAlign='center';ctx.textBaseline='middle';
    for(const label of svg.querySelectorAll('text')){ctx.fillStyle=label.getAttribute('fill')||'#d7f3ff';ctx.fillText(label.textContent,Number(label.getAttribute('x')),Number(label.getAttribute('y')))}ctx.restore();
    ctx.save();ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061019';ctx.shadowBlur=5;ctx.font='600 '+Math.max(9,Math.round(w/110))+'px system-ui,sans-serif';ctx.textAlign='left';ctx.fillText('ANATOMICAL ORIENTATION',x,h-size-29);ctx.restore();
  }
  function presentationFrame(maxWidth,forGif=false,showSection=false){
    const source=renderer.domElement,ratio=forGif?maxWidth/source.width:Math.min(1,maxWidth/source.width),w=Math.max(2,Math.round(source.width*ratio));
    const oldBackground=scene.background;scene.background=null;renderer.setClearColor(0x000000,0);renderer.render(scene,camera);
    const h=forGif?Math.round(w*9/16):Math.max(2,Math.round(source.height*ratio));
    const out=document.createElement('canvas');out.width=w;out.height=h;const ctx=out.getContext('2d',{willReadFrequently:true});if(forGif){ctx.fillStyle='#05080d';ctx.fillRect(0,0,w,h)}else ctx.clearRect(0,0,w,h);
    if(forGif&&showSection){
      const gap=Math.max(10,Math.round(w*.014)),panelX=Math.round(w*.64),leftW=panelX-gap,boxX=panelX,boxW=w-boxX-gap;
      const cropRatio=leftW/h;let cropH=source.height,cropW=cropH*cropRatio;
      if(cropW>source.width){cropW=source.width;cropH=cropW/cropRatio}
      ctx.drawImage(source,(source.width-cropW)/2,(source.height-cropH)/2,cropW,cropH,0,0,leftW,h);
      ctx.fillStyle='rgba(9,22,31,.94)';ctx.fillRect(boxX,gap,boxW,h-2*gap);ctx.strokeStyle='rgba(145,218,237,.45)';ctx.lineWidth=1;ctx.strokeRect(boxX+.5,gap+.5,boxW-1,h-2*gap-1);
      const pad=Math.max(12,Math.round(w*.013)),textX=boxX+pad,innerW=boxW-2*pad;
      ctx.fillStyle='#9ee9f4';ctx.font='700 '+Math.max(10,Math.round(w/95))+'px system-ui,sans-serif';ctx.fillText('2D SECTION · M07 MODEL',textX,gap+pad+3);
      ctx.fillStyle='#d9eaf2';ctx.font=Math.max(9,Math.round(w/115))+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+21,innerW);
      const addressY=h-gap-Math.max(94,Math.round(h*.17));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+34,imageH=Math.max(20,Math.min(innerW*section.height/section.width,addressY-imageTop-12)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(section,textX+(innerW-imageW)/2,imageTop+(addressY-12-imageTop-imageH)/2,imageW,imageH);
      ctx.strokeStyle='rgba(145,218,237,.24)';ctx.beginPath();ctx.moveTo(textX,addressY-12);ctx.lineTo(textX+innerW,addressY-12);ctx.stroke();
      ctx.fillStyle='#9ee9f4';ctx.font='700 '+Math.max(9,Math.round(w/116))+'px system-ui,sans-serif';ctx.fillText('SECTION PLANE ADDRESS · M07 LOCAL',textX,addressY+3);
      const az=sectionAngle*Math.PI/180,el=sectionTilt*Math.PI/180,nx=Math.cos(el)*Math.cos(az),ny=Math.cos(el)*Math.sin(az),nz=Math.sin(el);
      ctx.fillStyle='#d9eaf2';ctx.font=Math.max(9,Math.round(w/130))+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText('Frame +R / +V / +Cr · mm',textX,addressY+23,innerW);
      ctx.fillText('n = ('+[nx,ny,nz].map(v=>v.toFixed(3)).join(', ')+')',textX,addressY+41,innerW);
      ctx.fillText('n · x = '+sectionDepth.toFixed(3)+' mm',textX,addressY+59,innerW);
      drawCompactCompass(ctx,leftW,h,true);
    }else{
      if(forGif){const ar=w/h;let ch=source.height,cw=ch*ar;if(cw>source.width){cw=source.width;ch=cw/ar}ctx.drawImage(source,(source.width-cw)/2,(source.height-ch)/2,cw,ch,0,0,w,h)}else ctx.drawImage(source,0,0,w,h);drawCompactCompass(ctx,w,h);
      if(sectionPlane.visible){ctx.save();ctx.font=Math.max(10,Math.round(w/75))+'px system-ui,sans-serif';ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061018';ctx.shadowBlur=5;const line='M07 · az '+Number(angle.value).toFixed(0)+'° · el '+Number(angle2.value).toFixed(0)+'° · d '+Number(depth.value).toFixed(2)+' mm';ctx.fillText(line,Math.max(12,w*.02),h-Math.max(14,h*.04));ctx.restore()}
    }
    if(wholeAlveolarGroup.visible){ctx.save();ctx.fillStyle='#d7f3ff';ctx.shadowColor='#061018';ctx.shadowBlur=5;ctx.font=Math.max(9,Math.round(w/110))+'px system-ui,sans-serif';ctx.fillText('SCHEMATIC AIRSPACES · ILLUSTRATIVE',Math.max(12,w*.016),Math.max(18,h*.04));ctx.restore()}
    ctx.save();const credit='Arash Bagherabadi (2026) · github.com/arashabadi',fs=Math.max(14,Math.round(w/105)),cx=forGif&&showSection?w-Math.max(18,Math.round(w*.02)):w/2,cy=h-Math.max(11,Math.round(h*.02));ctx.font='600 '+fs+'px system-ui,sans-serif';ctx.textAlign=forGif&&showSection?'right':'center';if(!forGif){const tw=ctx.measureText(credit).width;ctx.fillStyle='rgba(5,8,13,.86)';ctx.fillRect(cx-tw/2-8,cy-fs-4,tw+16,fs+11)}ctx.fillStyle='rgba(214,237,246,.92)';ctx.fillText(credit,cx,cy);ctx.restore();ctx.save();ctx.textAlign='left';ctx.font='700 '+Math.max(14,Math.round(w/100))+'px system-ui,sans-serif';ctx.fillStyle='#d7f3ff';ctx.shadowColor='#05080d';ctx.shadowBlur=6;ctx.fillText('Mouse lung explorer · v3.1',Math.max(18,Math.round(w*.02)),Math.max(22,Math.round(h*.045)));ctx.restore();
    scene.background=oldBackground;renderer.render(scene,camera);return out;
  }
  const pngSetup=document.getElementById('pngSetup'),pngQuality=document.getElementById('pngQuality');
  capturePngButton.setAttribute('aria-label','Export figure or transparent PNG');capturePngButton.title='Export figure or transparent PNG';
  capturePngButton.addEventListener('click',()=>{pngSetup.hidden=false;document.getElementById('gifSetup').hidden=true;document.getElementById('videoSetup').hidden=true;pngQuality.focus()});
  document.getElementById('pngCancel').addEventListener('click',()=>pngSetup.hidden=true);
  document.getElementById('pngExport').addEventListener('click',()=>{const figure=pngQuality.value==='figure',name=figure?'mouse-lung-M07-v3.1-figure.png':'mouse-lung-M07-v3.1-transparent.png';pngSetup.hidden=true;saveCanvasPng(figure?presentationFrame(3200,true,clipping):presentationFrame(2200),name)});
  const gifSetup=document.getElementById('gifSetup'),gifQuality=document.getElementById('gifQuality'),gifEstimate=document.getElementById('gifEstimate'),gifTimer=document.getElementById('gifTimer');
  const gifPresets={compact:{width:640,fps:12},clear:{width:960,fps:15},slide:{width:1280,fps:20}};
  let gifRecording=false,gifExporting=false,gifStart=0,gifLast=-Infinity,gifModule=null,gifOptions=gifPresets.slide,gifEncoder=null,gifFrameCount=0,gifLayoutClipped=false;
  function updateGifEstimate(){
    const option=gifPresets[gifQuality.value],w=option.width,h=clipping?Math.round(w*9/16):Math.round(w*renderer.domElement.height/renderer.domElement.width),megabytes=(w*h*4/1048576).toFixed(1);
    gifEstimate.textContent='No time limit · click the red record icon to stop. Frames are encoded as you record ('+megabytes+' MB raw per frame); long GIFs and higher quality use more local CPU and storage.';
  }
  gifQuality.addEventListener('change',updateGifEstimate);
  document.getElementById('gifCancel').addEventListener('click',()=>gifSetup.hidden=true);
  gifSetup.addEventListener('keydown',event=>{if(event.key==='Escape')gifSetup.hidden=true});
  function recordTick(now){
    if(!gifRecording)return;
    if(now-gifLast>=1000/gifOptions.fps){
      try{
        const frame=presentationFrame(gifOptions.width,true,gifLayoutClipped),pixels=frame.getContext('2d',{willReadFrequently:true}).getImageData(0,0,frame.width,frame.height).data;
        for(let p=0;p<pixels.length;p+=4)pixels[p+3]=pixels[p+3]<128?0:255;
        const palette=gifModule.quantize(pixels,256,{format:'rgba4444',oneBitAlpha:true,clearAlpha:true}),index=gifModule.applyPalette(pixels,palette,'rgba4444'),transparentIndex=palette.findIndex(color=>color[3]===0);
        const delay=gifFrameCount?Math.max(20,Math.min(1000,Math.round((now-gifLast)/10)*10)):Math.round(1000/gifOptions.fps/10)*10;
        gifEncoder.writeFrame(index,frame.width,frame.height,{palette,delay,transparent:transparentIndex>=0,transparentIndex:Math.max(0,transparentIndex),dispose:2});
        gifFrameCount++;gifLast=now;const elapsed=Math.floor((now-gifStart)/1000);gifTimer.textContent=String(Math.floor(elapsed/60)).padStart(2,'0')+':'+String(elapsed%60).padStart(2,'0');gifButton.title='Stop recording · '+gifTimer.textContent+' · '+gifFrameCount+' frames';
      }catch(error){console.error('GIF frame failed',error);gifRecording=false;gifEncoder=null;gifTimer.hidden=true;gifButton.setAttribute('aria-pressed','false');window.alert('GIF capture stopped: '+error.message);return}
    }
    requestAnimationFrame(recordTick);
  }
  function finishGif(){
    if(!gifRecording)return;gifRecording=false;gifExporting=true;gifTimer.hidden=true;gifButton.setAttribute('aria-pressed','false');gifButton.setAttribute('aria-label','Choose GIF recording quality');document.getElementById('toggleClip').disabled=false;
    if(!gifFrameCount){gifEncoder=null;gifExporting=false;gifButton.title='No frames recorded';return}
    try{gifButton.title='Finalizing GIF…';gifEncoder.finish();const blob=new Blob([gifEncoder.bytes()],{type:'image/gif'}),url=URL.createObjectURL(blob),link=document.createElement('a');link.href=url;link.download='mouse-lung-M07-v3.1-'+gifQuality.value+'.gif';document.body.appendChild(link);link.click();link.remove();setTimeout(()=>URL.revokeObjectURL(url),10000);gifButton.title='GIF ready · '+gifFrameCount+' frames · '+(blob.size/1048576).toFixed(1)+' MB'}catch(error){console.error('GIF export failed',error);gifButton.title='GIF export failed';window.alert('GIF export failed: '+error.message)}finally{gifEncoder=null;gifExporting=false;gifFrameCount=0}
  }
  gifButton.addEventListener('click',()=>{
    if(gifRecording){finishGif();return}
    if(gifExporting)return;
    gifSetup.hidden=false;updateGifEstimate();gifQuality.focus();
  });
  document.getElementById('gifStart').addEventListener('click',async()=>{
    if(gifRecording||gifExporting)return;
    gifOptions=gifPresets[gifQuality.value];gifSetup.hidden=true;gifButton.title='Loading embedded GIF encoder…';
    try{gifModule=gifModule||await import('../assets/gifenc.esm.js')}catch(error){console.error(error);gifButton.title='GIF encoder unavailable';window.alert('Could not load the embedded GIF encoder.');return}
    gifEncoder=gifModule.GIFEncoder();gifFrameCount=0;gifLayoutClipped=clipping;if(gifLayoutClipped)drawSlice();document.getElementById('toggleClip').disabled=true;
    gifRecording=true;gifStart=performance.now();gifLast=-Infinity;gifTimer.hidden=false;gifTimer.textContent='00:00';gifButton.setAttribute('aria-pressed','true');gifButton.setAttribute('aria-label','Stop GIF recording');requestAnimationFrame(recordTick);
  });
  const videoButton=document.getElementById('recordLungVideo'),videoSetup=document.getElementById('videoSetup'),videoNote=document.getElementById('videoFormatNote');
  const videoTypes=['video/mp4;codecs=avc1.42E01E','video/mp4;codecs=avc1','video/webm;codecs=vp9','video/webm;codecs=vp8'];
  const videoMime=typeof MediaRecorder==='undefined'?null:videoTypes.find(type=>MediaRecorder.isTypeSupported(type));
  let videoRecorder=null,videoStream=null,videoCanvas=null,videoChunks=[],videoRaf=0,videoRecording=false,videoLast=-Infinity,videoClip=false;
  videoNote.textContent=videoMime?videoMime.startsWith('video/mp4')?'MP4 / H.264 is available here. 16:9, 30 fps; check your target journal’s file-size limit.':'This browser offers WebM, not MP4 / H.264. Check whether your target journal accepts WebM.':'This browser does not support local canvas video recording.';
  document.getElementById('videoStart').disabled=!videoMime;
  videoButton.addEventListener('click',()=>{
    if(videoRecording){videoRecording=false;cancelAnimationFrame(videoRaf);videoRecorder.stop();videoButton.setAttribute('aria-pressed','false');videoButton.setAttribute('aria-label','Set up supplementary video');document.getElementById('toggleClip').disabled=false;gifButton.disabled=false;capturePngButton.disabled=false;return}
    videoSetup.hidden=false;gifSetup.hidden=true;pngSetup.hidden=true;document.getElementById('videoQuality').focus();
  });
  document.getElementById('videoCancel').addEventListener('click',()=>videoSetup.hidden=true);
  function videoTick(now){
    if(!videoRecording)return;
    if(now-videoLast>=1000/30){try{const frame=presentationFrame(videoCanvas.width,true,videoClip);videoCanvas.getContext('2d').drawImage(frame,0,0);videoLast=now}catch(error){console.error('Video frame failed',error);videoButton.click();return}}
    videoRaf=requestAnimationFrame(videoTick);
  }
  document.getElementById('videoStart').addEventListener('click',()=>{
    if(!videoMime||videoRecording||gifRecording||gifExporting)return;
    try{
      videoClip=clipping;if(videoClip)drawSlice();const width=Number(document.getElementById('videoQuality').value);
      videoCanvas=document.createElement('canvas');videoCanvas.width=width;videoCanvas.height=Math.round(width*9/16);
      videoCanvas.getContext('2d').drawImage(presentationFrame(width,true,videoClip),0,0);
      videoStream=videoCanvas.captureStream(30);videoRecorder=new MediaRecorder(videoStream,{mimeType:videoMime,videoBitsPerSecond:width===1920?8000000:4500000});videoChunks=[];
      videoRecorder.ondataavailable=e=>{if(e.data.size)videoChunks.push(e.data)};
      videoRecorder.onerror=e=>{console.error('Video recorder error',e);videoButton.title='Video recording failed'};
      videoRecorder.onstop=()=>{const actualType=videoRecorder.mimeType||videoMime,ext=actualType.startsWith('video/mp4')?'mp4':'webm',blob=new Blob(videoChunks,{type:actualType}),url=URL.createObjectURL(blob),link=document.createElement('a');link.href=url;link.download='mouse-lung-M07-v3.1-supplementary.'+ext;document.body.appendChild(link);link.click();link.remove();setTimeout(()=>URL.revokeObjectURL(url),10000);videoButton.title='Video ready · '+(blob.size/1048576).toFixed(1)+' MB · '+ext.toUpperCase();videoStream.getTracks().forEach(track=>track.stop());videoChunks=[];videoRecorder=null;videoCanvas=null};
      videoSetup.hidden=true;videoRecorder.start(1000);videoRecording=true;videoLast=-Infinity;videoButton.setAttribute('aria-pressed','true');videoButton.setAttribute('aria-label','Stop supplementary video');videoButton.title='Stop and download video';document.getElementById('toggleClip').disabled=true;gifButton.disabled=true;capturePngButton.disabled=true;videoRaf=requestAnimationFrame(videoTick);
    }catch(error){console.error('Video recording unavailable',error);videoSetup.hidden=false;window.alert('Video recording could not start in this browser: '+error.message)}
  });

"""
old_capture = """  document.getElementById('saveLungPng').addEventListener('click',()=>{
    document.getElementById('saveLungPng').textContent='Preparing PNG…';
    renderer.render(scene,camera);
    if(clipping)drawSlice();
    const section=clipping?document.getElementById('sliceCanvas'):null;
    const schematic=wholeAlveolarGroup.visible;
    const label=schematic?'schematic airspaces enabled; illustrative size/count':'measured M07 gross geometry';
    compositeCapture(renderer.domElement,section,'M07 anatomical reference · version 3.0',label,'mouse-lung-v3.0.png');
  });"""
replace_once(old_capture, capture_js)
# Inline the small vendored encoder as a data-URL module: file:// viewers cannot fetch sibling ES modules reliably.
import base64
encoder = (ROOT / 'assets/gifenc.esm.js').read_bytes()
encoded = base64.b64encode(encoder).decode('ascii')
replace_once("import('../assets/gifenc.esm.js')", f"import('data:text/javascript;base64,{encoded}')")
replace_once("document.getElementById('saveLungPng').textContent='Save PNG';", '')
replace_once('controls.update();updateOrientationOverlay();updateSelectedLabel();', 'controls.update();updateOrientationOverlay();updateSelectedLabel();syncPlaneControls();')
print('Transparent PNG and GIF export patched')

Transparent PNG and GIF export patched


## 3. Versioned output and structural checks
The GIF encoder is vendored under `assets/` for the build and embedded as a data-URL ES module in the output HTML. This lets the viewer work when opened directly via `file://` and keeps GitHub Pages static. Interaction quality still requires a browser QA pass.

In [5]:
out = ROOT / 'viewer/mouse_lung_v3.1.html'
out.write_text(src)
assert 'alpha:true' in src and "import('data:text/javascript;base64," in src and 'gifEncoder.writeFrame' in src
assert 'Mouse lung explorer · v3.1' in src and 'Alveolar chicken-wire view' in src and 'gifPresets' in src and 'gifSetup' in src and 'SECTION PLANE ADDRESS' in src
assert '<output id="opacityOut">95%</output>' in src and 'opacity.dispatchEvent' in src
assert 'Ruberte et al. (2025)' in src and 'M07 local frame (mm)' in src
assert 'textContent=\'Preparing PNG…\'' not in src
assert 'Model coordinates (mm) & NAV base' not in src and 'Xenium section evidence' not in src and 'IF/Xenium' not in src
assert 'https://github.com/alopez789' in src and 'Current M07 plane · model coordinates (mm)' in src
assert 'https://arashabadi.github.io/' in src and 'Arash Bagherabadi (2026) · github.com/arashabadi' in src
assert "if(forGif){ctx.fillStyle='#05080d'" in src
assert 'MediaRecorder.isTypeSupported' in src and 'captureStream(30)' in src
assert 'Figure · 3200 px · dark 16:9' in src and 'Export figure or transparent PNG' in src
assert 'id="offsetPad"' in src and 'moveOffset(event)' in src
print('Built:', out, 'bytes:', out.stat().st_size, 'SHA256:', hashlib.sha256(out.read_bytes()).hexdigest())

Built: /Users/ashi/github/mouse-lung-explorer/viewer/mouse_lung_v3.1.html bytes: 44917898 SHA256: 3e5428b9308ef4922dbe1e6ed8151341ad0ede3a42c6384f265ff07fd17d4e61
